# Chapter 5: Formatting Output & Structured Responses

**Technique:** Output formatting and response prefilling

**Contents**
* [Lesson: Controlling Output Shape](#lesson)
* [Exercises](#exercises)
* Example Playground

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Controlling Output Shape

When you consume Claude's output in a program, not just display it to a human, you need the response to arrive in a predictable shape. Four techniques give you that control:

### 1. Explicit "no preamble" instructions

By default Claude may open with a sentence like "Sure, here's my review…". Tell it not to:

```js
const PROMPT = "Review this code. Skip any preamble, output only the review.\nconst x = arr[arr.length];";
```

This works for any format: JSON, markdown tables, bullet lists. The instruction travels inside the user message, so no extra API parameters are needed.

### 2. XML tags for structured prose

Ask Claude to wrap sections in XML tags and parse (or highlight) them downstream:

```js
const PROMPT = "Review this function and wrap your answer in <review></review> tags.\nconst x = arr[arr.length];";
```

Tags give you a reliable string boundary to extract with a regex or a simple `split`. They also let you request *multiple named sections* in one call, for example `<summary>` and `<risks>`.

### 3. Structured outputs via `output_config.format`

For machine readable payloads, use the `output_config` parameter with a JSON Schema. The SDK guarantees the response text is valid JSON that conforms to your schema:

```js
const msg = await client.messages.create({
  model: MODEL,
  max_tokens: 1024,
  temperature: 0,
  messages: [{ role: "user", content: "..." }],
  output_config: {
    format: {
      type: "json_schema",
      schema: {
        type: "object",
        properties: {
          summary: { type: "string" },
          issues: { type: "array", items: { type: "string" } },
        },
        required: ["summary", "issues"],
        additionalProperties: false,
      },
    },
  },
});
const data = JSON.parse(msg.content[0].text);
```

Important details:
* Call `client.messages.create` directly (not the `getCompletion` helper) so you can pass `output_config`.
* Always include `additionalProperties: false`; without it the schema is underspecified and the model may add unexpected fields.
* The response is still a string (`msg.content[0].text`); call `JSON.parse` to get the object.

### 4. Stop sequences

Pass a `stop_sequences` array to truncate the response at a sentinel string. Useful when you want only the first section of a longer potential output:

```js
const msg = await client.messages.create({
  model: MODEL,
  max_tokens: 1024,
  messages: [{ role: "user", content: "..." }],
  stop_sequences: ["</review>"],
});
```

### Historical note: assistant turn prefilling

Older Anthropic models accepted a partial `assistant` turn in the messages array, letting you "speak for Claude" and force a specific opening. On current models (Sonnet 4.6+) that pattern returns a **400 Bad Request**, so the four techniques above replace it entirely.

In [ ]:
console.log(await getCompletion("Review this line and wrap your answer in <review></review> tags: const x = arr[arr.length];"));

In [ ]:
const msg = await client.messages.create({
  model: MODEL,
  max_tokens: 1024,
  temperature: 0,
  messages: [{ role: "user", content: "Extract issues from this review note: 'off-by-one in lastItem; also missing null check'." }],
  output_config: {
    format: {
      type: "json_schema",
      schema: {
        type: "object",
        properties: {
          summary: { type: "string" },
          issues: { type: "array", items: { type: "string" } },
        },
        required: ["summary", "issues"],
        additionalProperties: false,
      },
    },
  },
});
const data = JSON.parse(msg.content[0].text);
console.log(data);

## Exercises

### Exercise 5.1: Wrap a code review in `<review>` tags

**Scenario**: your CI pipeline parses Claude's code review output by looking for a `<review>` XML tag. The grader simulates that parser.

**Task**: write a prompt that asks Claude to review this line, `const last = arr[arr.length];`, and wrap its answer in `<review></review>` tags.

**Success criteria**: Claude's response contains the string `<review>`.

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const PROMPT = "[Replace this text]";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => includesAny(text, ["<review>"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_5_1_hint } from "../hints.js";
console.log(exercise_5_1_hint);

### Exercise 5.2: Two delimited sections: `<summary>` and `<risks>`

**Scenario**: a security review tool expects two separate XML sections (a brief summary and a risk list) so each can be routed to a different downstream handler.

**Task**: write a prompt that asks Claude to review the code snippet `fetch(userInput)` and return **two sections**: one wrapped in `<summary></summary>` tags and one wrapped in `<risks></risks>` tags.

**Success criteria**: Claude's response contains both `<summary>` and `<risks>`.

In [ ]:
import { includesAll, grade } from "../lib/grading.js";

const PROMPT = "[Replace this text]";

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => includesAll(text, ["<summary>", "<risks>"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_5_2_hint } from "../hints.js";
console.log(exercise_5_2_hint);

### Exercise 5.3: Structured output with `output_config.format`

**Scenario**: a code analysis pipeline needs machine readable output it can pass directly to a database insert, with no postprocessing regex and no fragile text parsing.

**Task**: fill in the `schema` object so Claude's response parses to a JavaScript object with:
* `summary`, a `string` field
* `issues`, an `array` of strings

The `USER` message and the API call are already wired up; you only need to replace the `schema` placeholder.

**Success criteria**: `JSON.parse(msg.content[0].text)` produces an object where `typeof obj.summary === "string"` and `Array.isArray(obj.issues)` are both true.

In [ ]:
import { grade } from "../lib/grading.js";

const USER = "Summarize and list the problems in: 'no input validation; SQL string concatenation; no tests'.";
const schema = {
  // [Replace this object so the output has: summary (string) and issues (array of strings)]
};

const msg = await client.messages.create({
  model: MODEL, max_tokens: 1024, temperature: 0,
  messages: [{ role: "user", content: USER }],
  output_config: { format: { type: "json_schema", schema } },
});
let parsed = null;
try { parsed = JSON.parse(msg.content[0].text); } catch (_) { /* leave null */ }
const gradeExercise = (obj) => !!obj && typeof obj.summary === "string" && Array.isArray(obj.issues);
grade(msg.content[0].text, gradeExercise(parsed));

In [ ]:
import { exercise_5_3_hint } from "../hints.js";
console.log(exercise_5_3_hint);

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Trying to prefill the assistant turn**: passing `{ role: "assistant", content: "..." }` as the last message returns a **400 Bad Request** on Sonnet 4.6+. Use explicit instructions or `output_config.format` instead.
* **Forgetting `additionalProperties: false`**: without it the JSON Schema is underspecified. Claude may add extra fields that break downstream code expecting exactly the fields you declared.
* **Not calling `JSON.parse`**: `msg.content[0].text` is always a string. Even when structured output is enabled, you must parse it yourself before accessing properties.

**Stretch challenge**

Extend the Exercise 5.3 schema to include a `severity` field with an `enum` constraint (`["low", "medium", "high"]`). Add `"severity"` to the `required` array, rebuild, and observe how Claude populates it.

**Reflect**: when is structured output (`output_config.format`) better than asking Claude to return JSON in free text and parsing it with a regex? Consider error handling, schema evolution, and the cost of a malformed response in a production pipeline.

## Example Playground

Use the cells below to experiment freely. Try combining XML tags with `output_config.format`, add stop sequences, or request additional schema fields.

In [ ]:
// Experiment: structured output with an extra "severity" field
const schema = {
  type: "object",
  properties: {
    summary: { type: "string" },
    issues: { type: "array", items: { type: "string" } },
    severity: { type: "string", enum: ["low", "medium", "high"] },
  },
  required: ["summary", "issues", "severity"],
  additionalProperties: false,
};

const msg = await client.messages.create({
  model: MODEL,
  max_tokens: 1024,
  temperature: 0,
  messages: [{ role: "user", content: "Analyze this code: fetch(window.location.hash.slice(1))" }],
  output_config: { format: { type: "json_schema", schema } },
});
console.log(JSON.parse(msg.content[0].text));